# 03 — Censo 2022: NBI y Etnia

**Fuente:** VIII Censo de Población y VII de Vivienda 2022 — INEC  
**Archivos:**
- `raw/CENSO_NBI/1.2.csv` — NBI por provincia, cantón, parroquia y área
- `raw/CENSO_NBI/2.2.csv` — NBI por provincia, cantón, parroquia y etnia
- `raw/CENSO_ETNIA/1.2.csv` — Población por autoidentificación étnica

**Objetivo:** Extraer % NBI y % población indígena por provincia para los cruces con mortalidad por diabetes y compras SERCOP.

In [1]:
import pandas as pd
import unicodedata
from pathlib import Path

pd.set_option('display.float_format', '{:,.2f}'.format)

RAW_NBI = Path('../raw/CENSO_NBI')
RAW_ETNIA = Path('../raw/CENSO_ETNIA')
PROCESSED_DIR = Path('../processed')
PROCESSED_DIR.mkdir(exist_ok=True)

SKIP = 12  # filas de metadata antes de los datos

def normalizar_nombre(s):
    """Quita tildes y convierte a mayúsculas para hacer join con otros datasets."""
    s = str(s).strip().upper()
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

## 1. NBI Tabla 1.2 — por provincia, cantón, parroquia y área

In [2]:
nbi_raw = pd.read_csv(
    RAW_NBI / '1.2.csv',
    encoding='latin-1',
    header=None,
    skiprows=SKIP
)

# Asignar nombres a columnas relevantes
nbi_raw.columns = [
    'idx', 'provincia', 'canton', 'parroquia', 'area',
    'pob_total',
    'no_pobres_h', 'no_pobres_m', 'no_pobres_total',
    'pobres_h', 'pobres_m', 'pobres_total'
]

print(f'Shape: {nbi_raw.shape}')
nbi_raw.head(8)

Shape: (3045, 12)


,idx,provincia,canton,parroquia,area,pob_total,no_pobres_h,no_pobres_m,no_pobres_total,pobres_h,pobres_m,pobres_total
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Total Nacional,Nacional,Nacional,Nacional,"16,884,398","4,912,517","5,258,131","10,170,648","3,297,557","3,416,193","6,713,750"
2,NaN,Total Nacional,Nacional,Nacional,Urbana,"10,653,064","3,742,374","4,025,567","7,767,941","1,393,223","1,491,900","2,885,123"
3,NaN,Total Nacional,Nacional,Nacional,Rural,"6,231,334","1,170,143","1,232,564","2,402,707","1,904,334","1,924,293","3,828,627"
4,NaN,Azuay,Total Azuay,Total Azuay,Total Azuay,"799,010","280,234","313,057","593,291","94,490","111,229","205,719"
5,NaN,Azuay,Total Azuay,Total Azuay,Urbana,"418,975","176,733","196,299","373,032","21,665","24,278","45,943"
6,NaN,Azuay,Total Azuay,Total Azuay,Rural,"380,035","103,501","116,758","220,259","72,825","86,951","159,776"
7,NaN,Azuay,Cuenca,Total Cuenca,Total Cuenca,"593,830","225,840","250,245","476,085","54,457","63,288","117,745"


In [3]:
# Limpiar: eliminar filas de metadata residual (nulas en provincia)
nbi = nbi_raw[nbi_raw['provincia'].notna()].copy()

# Convertir columnas numéricas (vienen con comas de miles)
for col in ['pob_total', 'no_pobres_total', 'pobres_total', 'no_pobres_h', 'no_pobres_m', 'pobres_h', 'pobres_m']:
    nbi[col] = pd.to_numeric(nbi[col].astype(str).str.replace(',', ''), errors='coerce')

print(f'Filas con datos: {len(nbi)}')
print(f'Nulos en pob_total: {nbi["pob_total"].isna().sum()}')
nbi.head(6)

Filas con datos: 3043
Nulos en pob_total: 3


,idx,provincia,canton,parroquia,area,pob_total,no_pobres_h,no_pobres_m,no_pobres_total,pobres_h,pobres_m,pobres_total
1,NaN,Total Nacional,Nacional,Nacional,Nacional,"16,884,398.00","4,912,517.00","5,258,131.00","10,170,648.00","3,297,557.00","3,416,193.00","6,713,750.00"
2,NaN,Total Nacional,Nacional,Nacional,Urbana,"10,653,064.00","3,742,374.00","4,025,567.00","7,767,941.00","1,393,223.00","1,491,900.00","2,885,123.00"
3,NaN,Total Nacional,Nacional,Nacional,Rural,"6,231,334.00","1,170,143.00","1,232,564.00","2,402,707.00","1,904,334.00","1,924,293.00","3,828,627.00"
4,NaN,Azuay,Total Azuay,Total Azuay,Total Azuay,"799,010.00","280,234.00","313,057.00","593,291.00","94,490.00","111,229.00","205,719.00"
5,NaN,Azuay,Total Azuay,Total Azuay,Urbana,"418,975.00","176,733.00","196,299.00","373,032.00","21,665.00","24,278.00","45,943.00"
6,NaN,Azuay,Total Azuay,Total Azuay,Rural,"380,035.00","103,501.00","116,758.00","220,259.00","72,825.00","86,951.00","159,776.00"


### 1.1 NBI por provincia (total, todas las áreas)

In [4]:
# Filas de totales provinciales: canton, parroquia y area empiezan con 'Total'
mask_prov = (
    nbi['canton'].astype(str).str.startswith('Total') &
    nbi['parroquia'].astype(str).str.startswith('Total') &
    nbi['area'].astype(str).str.startswith('Total')
)
nbi_prov_all = nbi[mask_prov].copy()
nbi_prov_all = nbi_prov_all[nbi_prov_all['provincia'] != 'Total Nacional']
nbi_prov_all['provincia'] = nbi_prov_all['provincia'].apply(normalizar_nombre)

# Garantizar 1 fila por provincia: la de mayor pob_total (siempre es el total, no urbana/rural)
nbi_prov_all = nbi_prov_all.sort_values('pob_total', ascending=False)
nbi_prov_clean = nbi_prov_all.drop_duplicates('provincia', keep='first').copy()

nbi_prov_clean['pct_nbi'] = (nbi_prov_clean['pobres_total'] / nbi_prov_clean['pob_total'] * 100).round(1)

df_nbi_prov = nbi_prov_clean[['provincia', 'pob_total', 'pobres_total', 'pct_nbi']]\
    .sort_values('pct_nbi', ascending=False).reset_index(drop=True)

print(f'Provincias: {len(df_nbi_prov)}')
df_nbi_prov

Provincias: 24


,provincia,pob_total,pobres_total,pct_nbi
0,ORELLANA,"181,995.00","125,005.00",68.70
1,MORONA SANTIAGO,"191,813.00","125,461.00",65.40
2,ESMERALDAS,"550,425.00","346,609.00",63.00
3,NAPO,"131,082.00","80,894.00",61.70
4,LOS RIOS,"897,111.00","550,670.00",61.40
5,MANABI,"1,589,104.00","955,376.00",60.10
6,SUCUMBIOS,"197,969.00","117,250.00",59.20
7,BOLIVAR,"198,533.00","116,266.00",58.60
8,PASTAZA,"111,532.00","59,036.00",52.90
9,ZAMORA CHINCHIPE,"110,440.00","55,335.00",50.10


### 1.2 NBI por cantón (para Cruce 2 si granularidad parroquial tiene subregistro)

In [5]:
# Filas cantonales: canton NO empieza con 'Total', parroquia y area sí
mask_cant = (
    ~nbi['canton'].astype(str).str.startswith('Total') &
    nbi['parroquia'].astype(str).str.startswith('Total') &
    nbi['area'].astype(str).str.startswith('Total')
)
nbi_cant = nbi[mask_cant].copy()
nbi_cant['pct_nbi'] = (nbi_cant['pobres_total'] / nbi_cant['pob_total'] * 100).round(1)
nbi_cant['provincia'] = nbi_cant['provincia'].apply(normalizar_nombre)
nbi_cant['canton'] = nbi_cant['canton'].str.strip()

df_nbi_cant = nbi_cant[['provincia', 'canton', 'pob_total', 'pobres_total', 'pct_nbi']]\
    .reset_index(drop=True)

print(f'Cantones: {len(df_nbi_cant)}')
df_nbi_cant.head(10)

Cantones: 221


,provincia,canton,pob_total,pobres_total,pct_nbi
0,AZUAY,Cuenca,"593,830.00","117,745.00",19.80
1,AZUAY,Girón,"12,173.00","4,031.00",33.10
2,AZUAY,Gualaceo,"43,168.00","18,381.00",42.60
3,AZUAY,Nabón,"14,758.00","10,671.00",72.30
4,AZUAY,Paute,"26,721.00","8,696.00",32.50
5,AZUAY,Pucará,"9,668.00","6,237.00",64.50
6,AZUAY,San Fernando,"3,738.00","1,171.00",31.30
7,AZUAY,Santa Isabel,"20,713.00","7,756.00",37.40
8,AZUAY,Sígsig,"24,989.00","13,089.00",52.40
9,AZUAY,Oña,"3,419.00","1,930.00",56.40


## 2. Etnia Tabla 1.2 — % población indígena por provincia

In [6]:
etnia_raw = pd.read_csv(
    RAW_ETNIA / '1.2.csv',
    encoding='latin-1',
    header=None,
    skiprows=SKIP
)

etnia_raw.columns = [
    'idx', 'provincia', 'canton', 'parroquia', 'sexo',
    'pob_total',
    'indigena', 'afroecuatoriano', 'negro', 'mulato',
    'montubio', 'mestizo', 'blanco', 'otro'
]

print(f'Shape: {etnia_raw.shape}')
etnia_raw.head(6)

Shape: (3865, 14)


,idx,provincia,canton,parroquia,sexo,pob_total,indigena,afroecuatoriano,negro,mulato,montubio,mestizo,blanco,otro
0,NaN,Total Nacional,Nacional,Nacional,Hombres,"8,252,523","633,328","167,838","114,333","121,837","668,713","6,354,515","180,449","11,510"
1,NaN,Total Nacional,Nacional,Nacional,Mujeres,"8,686,463","668,729","175,588","111,484","123,415","636,287","6,767,822","194,481","8,657"
2,NaN,Azuay,Total Azuay,Total Azuay,Total Azuay,"801,609","15,992","2,854","1,385","2,853","3,315","758,909","15,998",303
3,NaN,Azuay,Total Azuay,Total Azuay,Hombres,"376,419","7,669","1,538",844,"1,511","1,757","355,231","7,698",171
4,NaN,Azuay,Total Azuay,Total Azuay,Mujeres,"425,190","8,323","1,316",541,"1,342","1,558","403,678","8,300",132
5,NaN,Azuay,Cuenca,Total Cuenca,Total Cuenca,"596,101","8,740","2,156","1,061","2,175","2,481","565,885","13,315",288


In [7]:
etnia = etnia_raw[etnia_raw['provincia'].notna()].copy()

for col in ['pob_total', 'indigena', 'afroecuatoriano', 'negro', 'mulato', 'montubio', 'mestizo', 'blanco', 'otro']:
    etnia[col] = pd.to_numeric(etnia[col].astype(str).str.replace(',', ''), errors='coerce')

print(f'Filas con datos: {len(etnia)}')
print(f'sexo valores: {etnia["sexo"].dropna().unique()[:8]}')

Filas con datos: 3864
sexo valores: ['Hombres' 'Mujeres' 'Total Azuay' 'Total Cuenca' 'Total Baños'
 'Total Cumbe' 'Total Chaucha' 'Total Checa']


In [8]:
# Totales provinciales: canton, parroquia y sexo empiezan con 'Total'
mask_prov_e = (
    etnia['canton'].astype(str).str.startswith('Total') &
    etnia['parroquia'].astype(str).str.startswith('Total') &
    etnia['sexo'].astype(str).str.startswith('Total')
)
etnia_prov = etnia[mask_prov_e].copy()
etnia_prov = etnia_prov[etnia_prov['provincia'] != 'Total Nacional']

# Garantizar 1 fila por provincia: la de mayor pob_total (siempre el total, no hombres/mujeres)
etnia_prov = etnia_prov.sort_values('pob_total', ascending=False)\
    .drop_duplicates('provincia', keep='first')

etnia_prov['pct_indigena'] = (etnia_prov['indigena'] / etnia_prov['pob_total'] * 100).round(1)
etnia_prov['provincia'] = etnia_prov['provincia'].apply(normalizar_nombre)

df_etnia_prov = etnia_prov[['provincia', 'pob_total', 'indigena', 'pct_indigena']]\
    .sort_values('pct_indigena', ascending=False)\
    .reset_index(drop=True)

print(f'Provincias: {len(df_etnia_prov)}')
df_etnia_prov

Provincias: 24


,provincia,pob_total,indigena,pct_indigena
0,NAPO,"131,675.00","85,528.00",65.00
1,MORONA SANTIAGO,"192,508.00","112,722.00",58.60
2,PASTAZA,"111,915.00","56,887.00",50.80
3,CHIMBORAZO,"471,933.00","178,754.00",37.90
4,ORELLANA,"182,166.00","67,858.00",37.30
5,BOLIVAR,"199,078.00","58,727.00",29.50
6,IMBABURA,"469,879.00","131,592.00",28.00
7,COTOPAXI,"470,210.00","111,446.00",23.70
8,ZAMORA CHINCHIPE,"110,973.00","20,986.00",18.90
9,SUCUMBIOS,"199,014.00","32,403.00",16.30


In [9]:
df_censo_prov = df_nbi_prov.merge(df_etnia_prov[['provincia', 'pct_indigena', 'indigena']], on='provincia', how='left')
print(f'Shape: {df_censo_prov.shape}')
print(f'Provincias sin etnia: {df_censo_prov["pct_indigena"].isna().sum()}')
df_censo_prov

Shape: (24, 6)
Provincias sin etnia: 0


,provincia,pob_total,pobres_total,pct_nbi,pct_indigena,indigena
0,ORELLANA,"181,995.00","125,005.00",68.70,37.30,"67,858.00"
1,MORONA SANTIAGO,"191,813.00","125,461.00",65.40,58.60,"112,722.00"
2,ESMERALDAS,"550,425.00","346,609.00",63.00,3.40,"18,572.00"
3,NAPO,"131,082.00","80,894.00",61.70,65.00,"85,528.00"
4,LOS RIOS,"897,111.00","550,670.00",61.40,0.70,"5,960.00"
5,MANABI,"1,589,104.00","955,376.00",60.10,0.20,"2,928.00"
6,SUCUMBIOS,"197,969.00","117,250.00",59.20,16.30,"32,403.00"
7,BOLIVAR,"198,533.00","116,266.00",58.60,29.50,"58,727.00"
8,PASTAZA,"111,532.00","59,036.00",52.90,50.80,"56,887.00"
9,ZAMORA CHINCHIPE,"110,440.00","55,335.00",50.10,18.90,"20,986.00"


## 4. Guardar

In [10]:
out1 = PROCESSED_DIR / 'censo_nbi_provincia.csv'
df_censo_prov.to_csv(out1, index=False)
print(f'Guardado: {out1} ({len(df_censo_prov)} filas)')

out2 = PROCESSED_DIR / 'censo_nbi_canton.csv'
df_nbi_cant.to_csv(out2, index=False)
print(f'Guardado: {out2} ({len(df_nbi_cant)} filas)')

Guardado: ../processed/censo_nbi_provincia.csv (24 filas)
Guardado: ../processed/censo_nbi_canton.csv (221 filas)


## 5. Resumen ETL

| Concepto | Valor |
|---|---|
| Filas NBI 1.2 (con datos) | 3.043 |
| Provincias NBI extraídas | 24 |
| Cantones NBI extraídos | 221 |
| Provincias etnia extraídas | 24 |
| Outputs generados | censo_nbi_provincia.csv (24 filas), censo_nbi_canton.csv (221 filas) |

**Decisiones ETL:**
- Los archivos del censo tienen 12 filas de metadata antes de los datos — se usa `skiprows=12`
- Las cifras vienen con separador de miles (coma) — se limpia con `str.replace(',', '')`
- Se identifican filas provinciales filtrando donde `canton`, `parroquia` y `area`/`sexo` empiezan con 'Total'
- Se aplica `normalizar_nombre()` (unicodedata NFD) para eliminar tildes y estandarizar en mayúsculas, permitiendo join con otros datasets
- `pct_nbi = pobres_total / pob_total * 100` — hogares bajo la línea NBI sobre total de la población
- `pct_indigena = indigena / pob_total * 100` — calculado desde el archivo de etnia del Censo 2022
- Se usa `sort_values('pob_total', ascending=False).drop_duplicates('provincia', keep='first')` para garantizar exactamente 1 fila por provincia
- Se incluye nivel cantonal como fallback para el Cruce 2 si la granularidad parroquial tiene subregistro en defunciones